## Carregamendo do gabarito  
  

Com base no conteúdo rotulado nas duas bases BOS.xlsx e DECC.xlsx aqui é realizado o carregamento do gabarito.

In [ ]:
import os
import re
import unicodedata
import pandas as pd
from unidecode import unidecode

In [ ]:
import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords

In [ ]:
PATH_BASE_BOS = os.getenv('PATH_BASE_BOS')
PATH_BASE_DECCC = os.getenv('PATH_BASE_DECCC')
PATH_GABARITO = os.getenv('PATH_GABARITO')

### Carregando a Base

In [ ]:
df_origin = pd.read_excel(PATH_BASE_BOS, usecols=['n_bop', 'relato', 'NOMES']).dropna(subset=['NOMES']).reset_index()
df_new = pd.read_excel(PATH_BASE_DECCC, usecols=['n_bop', 'relato', 'nomes']).dropna(subset=['nomes']).reset_index()

df = pd.DataFrame({'n_bop':pd.concat([df_origin['n_bop'], df_new['n_bop']], ignore_index=True),
                    'relato':pd.concat([df_origin['relato'], df_new['relato']], ignore_index=True),
                    'nomes': pd.concat([df_origin['NOMES'], df_new['nomes']], ignore_index=True)})
df

In [ ]:
df = pd.read_csv(r'C:\Users\joaol\OneDrive\Desktop\MestradoBaseMédica\ProjetoDoutorado\BASES\base_dados_medica_com_id.csv', usecols=['relato', 'nomes']).dropna(subset=['nomes']).reset_index(drop=True)

### Pré-processamento nos relatos

In [ ]:
def preprocess(relato):
    def remove_stop_words(relato):
        # Remove stop words em português
        sw = stopwords.words('portuguese')
        sw = [unidecode(word).upper() for word in sw]
        return ' '.join([token for token in re.split("[, ]", relato) if len(token) > 2 and token not in sw])
    
    def clean_relato(relato):
        # Remoção de números e pontuação    
        relato = re.sub(r'\@(\w*\d?)', " ", relato)
        relato = re.sub(r'([A-z1-9]*_+[A-z1-9]*)|(_+[A-z1-9]*)|([A-z1-9]*_+)', " ", relato)
        relato = re.sub(r"((?:HTTPS?|WWW)(?:\:\/\/?)?[*a-zA-Z0-9\.\/\?\:@\-_=#]+[\.][a-zA-Z]{2,6}[a-zA-Z0-9\.\&\/\?\:@\-_=#]*)", " ", relato)
        relato = re.sub(r'(<[^<>]*>)|(&NBSP;)|(&nbsp;)', " ", relato)
        relato = re.sub(r"[^\w\s]", " ", relato)
        relato = re.sub(r"[\d]", " ", relato)
        relato = remove_stop_words(relato)
        return relato
    
    relato = clean_relato(relato)
    return relato

def normalize_text(text):
    if pd.isna(text):
        return text
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(c for c in text if not unicodedata.combining(c))
    return text.upper()

In [ ]:
df['relato'] = df['relato'].apply(normalize_text)
df['relato_tratado'] = df['relato'].apply(preprocess)

### Extração dos nomes da coluna de rótulos

In [ ]:
coluna_nomes_base = [re.split(r', | |,', df['nomes'][i]) for i in range(0,len(df))]

In [ ]:
def load_names_base(coluna_nomes_base):
    lista_nomes = []
    for value in coluna_nomes_base:
        for non in value:
            lista_nomes.append(non)
    print("Tamanho original antes da remoção de duplicados: ", len(lista_nomes))
    return set(lista_nomes)

In [ ]:
nomes = load_names_base(coluna_nomes_base)

In [ ]:
print("Tamanho após a remoção dos duplicados: ", len(nomes))

### Obtenção da coluna de relatos e transformar em tokens

In [ ]:
coluna_relato_base = [df['relato_tratado'][i].split(' ') for i in range(0,len(df))]

In [ ]:
def load_words_base(coluna_relato_base):
    lista_relatos = []
    for value in coluna_relato_base:
        for word in value:
            lista_relatos.append(word)
    print("Tamanho original antes da remoção de duplicados: ", len(lista_relatos))
    return set(lista_relatos)

In [ ]:
palavras = load_words_base(coluna_relato_base)

In [ ]:
print("Tamanho após a remoção dos duplicados: ", len(palavras))

### Veficação da existencia de dados repetidos

In [ ]:
df_nomes = pd.DataFrame({'nomes':list(nomes)})
df_palavras = pd.DataFrame({'palavras':list(palavras)})

### Removendo valores vazios nas celulas

In [ ]:
df_nomes = df_nomes.replace({'':None})
df_nomes = df_nomes.dropna()

In [ ]:
df_palavras = df_palavras.replace({'':None})
df_palavras = df_palavras.dropna()

In [ ]:
df_palavras

In [ ]:
df_palavras[df_palavras['palavras'].isin(df_nomes['nomes'].to_list())]

In [ ]:
df_palavras = df_palavras[~df_palavras['palavras'].isin(df_nomes['nomes'].to_list())].reset_index(drop=True)

In [ ]:
df_palavras[df_palavras['palavras'].isin(df_nomes['nomes'].to_list())]

In [ ]:
df_palavras

In [ ]:
df_nomes['y'] = 1
df_palavras['y'] = 0

In [ ]:
tokens = df_nomes['nomes'].to_list() + df_palavras['palavras'].to_list()

In [ ]:
tokens

In [ ]:
rotulos = df_nomes['y'].to_list() + df_palavras['y'].to_list()

In [ ]:
gabarito = pd.DataFrame({'palavra':tokens, 'y':rotulos}).dropna()

In [ ]:
gabarito

In [ ]:
gabarito = gabarito.drop_duplicates(subset=['palavra'])
gabarito

### Correção gab - removendo stopwords que sobraram

Alguns nomes possuem tokens "stopwords" como "AQUELES" ou "AQUELE" essa etapa é pra realizar a remoçao destes. Devido problemas que ocorreram no momento da avalização, os tokenizadores do Bert e Spacy removeram porém passou no nosso, por isso estes casos foram removidos.

In [ ]:
stopwords_list = stopwords.words('portuguese')
print("aquele" in stopwords_list)

In [ ]:
for i in gabarito['palavra'].to_list():
    if i.lower() in stopwords_list:
        print(i)

In [ ]:
gabarito_sem_stopword = gabarito[~gabarito['palavra'].str.lower().isin(stopwords_list)]

In [ ]:
for i in gabarito_sem_stopword['palavra'].to_list():
    if i.lower() in stopwords_list:
        print(i)

In [ ]:
gabarito_sem_stopword

In [ ]:
gabarito_sem_stopword.to_csv(PATH_GABARITO, index = False)

In [ ]:
# 1. Definimos o caminho absoluto exato para evitar erro de nível (../)
caminho_final = r'C:\Users\joaol\OneDrive\Desktop\MestradoBaseMédica\ProjetoDoutorado\BASES\experimento_corretor\gabarito_corretor_base_medica.csv'

In [ ]:
# 2. Pegamos o diretório pai (C:\...\BASES\experimento_corretor)
diretorio = os.path.dirname(caminho_final)

In [ ]:
# 4. Salvamos o seu DataFrame 'gabarito_sem_stopword'
gabarito_sem_stopword.to_csv(caminho_final, index=False)

print(f"🚀 Gabarito salvo com sucesso em: {caminho_final}")